<a href="https://colab.research.google.com/github/turnleftorgo/Deep_learning/blob/main/nlp_assignment_2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part1 Basic model & basic function

## Preparing data

In [ ]:
!pip install gensim

**Important !!! ** After running the cell above to install `gensim`, please restart the Colab runtime before proceeding. You can do this by selecting `Runtime > Restart session` from the menu.

In [ ]:
import gensim.downloader as api # For loading Word2Vec, GloVe, etc.
import torch

In [ ]:
## Requirements
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import string
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import gdown

!gdown --fuzzy https://drive.google.com/file/d/16T0hVR0o8T8pIknCOJhrKa--QIAJaoeF/view?usp=sharing




Downloading...
From: https://drive.google.com/uc?id=16T0hVR0o8T8pIknCOJhrKa--QIAJaoeF
To: /content/train.csv
100% 76.0M/76.0M [00:00<00:00, 153MB/s]


In [ ]:
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-z0-9/ .!?]+", r" ", s)
    return s

In [ ]:
SOS_token = 0
EOS_token = 1
UNK_token = 2


class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS", 2: "UNK"}
        self.n_words = 3

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [ ]:
import pandas as pd
import unicodedata, re, random, torch, torch.nn as nn, torch.nn.functional as F

def readCSV(file_path, src_col, tgt_col, reverse=False):
    print("Reading lines from CSV…")
    df = pd.read_csv(file_path)

    # Normalise & split into list of [src, tgt]
    pairs = [
        [normalizeString(df.loc[i, src_col]),
         normalizeString(df.loc[i, tgt_col])]
        for i in range(len(df))
    ]

    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang  = Lang(tgt_col)
        output_lang = Lang(src_col)
    else:
        input_lang  = Lang(src_col)
        output_lang = Lang(tgt_col)

    return input_lang, output_lang, pairs


In [ ]:
file_path  = "/content/train.csv",
src_col    = "Ingredients",
tgt_col    = "Recipe"

In [ ]:
MAX_LENGTH = 50


def filterPair(pair):
    """
    Keep only pairs where both source and target are shorter than MAX_LENGTH.
    """
    return (len(pair[0].split()) < MAX_LENGTH and
            len(pair[1].split()) < MAX_LENGTH)


def filterPairs(pairs):
    """
    Apply filterPair to every (src, tgt) and return only the ones that pass.
    """
    return [pair for pair in pairs if filterPair(pair)]

In [ ]:
def prepareDataCSV(file_path, src_col, tgt_col, reverse=False):
    """
    1. Load & normalize with readCSV
    2. Print original size
    3. Filter by MAX_LENGTH
    4. Build vocab with Lang.addSentence
    5. Print trimmed size & vocab counts
    """
    # 1. load
    input_lang, output_lang, pairs = readCSV(
    file_path  = "/content/train.csv",
    src_col    = "Ingredients",
    tgt_col    = "Recipe",
    reverse    = False
)
    print(f"Read {len(pairs)} sentence pairs")

    # 2. filter
    pairs = filterPairs(pairs)
    print(f"Trimmed to {len(pairs)} sentence pairs")

    # 3. build vocab
    print("Counting words...")
    for src, tgt in pairs:
        input_lang.addSentence(src)
        output_lang.addSentence(tgt)

    # 4. report
    print("Counted words:")
    print(f"{input_lang.name}: {input_lang.n_words}")
    print(f"{output_lang.name}: {output_lang.n_words}")
    return input_lang, output_lang, pairs

input_lang, output_lang, pairs = prepareDataCSV(file_path, src_col, tgt_col)
print(random.choice(pairs))

Reading lines from CSV…
Read 162899 sentence pairs
Trimmed to 93743 sentence pairs
Counting words...
Counted words:
Ingredients: 6845
Recipe: 9236
[' 1 large box orange jell o   1 pt . cottage cheese   3 cans mandarin oranges   2 cans pineapple chunks or tidbits   1 pt . cool whip ', ' mix dry jell o with cottage cheese .   add drained oranges and pineapple .   add cool whip .   fold all together carefully until well mixed .   refrigerate a while before serving . ']


In [ ]:
import time
import math


def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)


def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [ ]:
def indexesFromSentence(lang, sentence, unk_idx=UNK_token):

    return [lang.word2index.get(word, unk_idx)
            for word in sentence.split(' ')]


def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(-1, 1)


def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

In [ ]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np
%matplotlib inline

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)


## basic model

In [ ]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        '''
        GRU is a gated RNN variant that captures long-term dependencies more effectively with fewer parameters. GRU has the same output shape as a standard RNN when configured identically.
        '''
        self.gru = nn.GRU(hidden_size, hidden_size)

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output = embedded
        output, hidden = self.gru(output, hidden)
        return output, hidden



In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)


In [ ]:
teacher_forcing_ratio = 0.2


def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=MAX_LENGTH):
    encoder_hidden = encoder.initHidden()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)

    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

    loss = 0




    for ei in range(min(input_length,max_length)):
        encoder_output, encoder_hidden = encoder(
            input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    decoder_input = torch.tensor([[SOS_token]], device=device)

    decoder_hidden = encoder_hidden

    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False



    if use_teacher_forcing:
        # Teacher forcing: Feed the target as the next input

        for di in range(min(target_length,max_length)):

            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden)

            loss += criterion(decoder_output, target_tensor[di])


            decoder_input = target_tensor[di]  # Teacher forcing

    else:
        # Without teacher forcing: use its own predictions as the next input
        for di in range(min(target_length,max_length)):
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()  # detach from history as input

            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length


In [ ]:
def trainIters(encoder, decoder, n_iters, print_every=1000, plot_every=100, learning_rate=0.01):

    # this part is for set-up

    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)

    training_pairs = [tensorsFromPair(random.choice(pairs))
                      for i in range(n_iters)]
    criterion = nn.NLLLoss()


    # this part is for the main training loop

    for iter in range(1, n_iters + 1):                                              # n_iters means the total number of training iterations (or steps) you want to run.

        training_pair = training_pairs[iter - 1]

        input_tensor = training_pair[0]
        target_tensor = training_pair[1]                                            # target_tensor is the tensor representation of the correct output sentence.

        loss = train(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, criterion)      # here is one step to using train.

        print_loss_total += loss
        plot_loss_total += loss

        if iter % print_every == 0:

            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, iter / n_iters),
                                         iter, iter / n_iters * 100, print_loss_avg))

        if iter % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)


In [ ]:
def evaluate(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]

        encoder_hidden = encoder.initHidden()

        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(min(input_length,max_length)):
            encoder_output, encoder_hidden = encoder(input_tensor[ei],
                                                     encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

        decoder_hidden = encoder_hidden

        decoded_words = []
        decoder_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden)
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words

In [ ]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words= evaluate(encoder, decoder, pair[0])
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [ ]:
hidden_size = 256
encoder11 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder11 = DecoderRNN(hidden_size, output_lang.n_words).to(device)

n_iters = 15000
print_every = 1000
trainIters(encoder11, decoder11, n_iters = n_iters, print_every=print_every)


In [ ]:
evaluateRandomly(encoder11, decoder11)

In [ ]:
checkpoint1 = {
    'encoder_state': encoder11.state_dict(),
    'decoder_state': decoder11.state_dict(),

}

torch.save(checkpoint1, 'seq2seq_checkpoint1.pth')

# Part 2 with attention

In [ ]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1, max_length=MAX_LENGTH):
        super(AttnDecoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.dropout_p = dropout_p
        self.max_length = max_length

        self.embedding = nn.Embedding(self.output_size, self.hidden_size)
        self.dropout = nn.Dropout(self.dropout_p)
        self.gru = nn.GRU(self.hidden_size, self.hidden_size)
        self.out = nn.Linear(self.hidden_size*2, self.output_size)

    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        embedded = self.dropout(embedded)

        _, hidden = self.gru(embedded, hidden)

        attn_weights = F.softmax(torch.bmm(hidden, encoder_outputs.T.unsqueeze(0)),dim=-1)
        attn_output = torch.bmm(attn_weights, encoder_outputs.unsqueeze(0))

        concat_output = torch.cat((attn_output[0], hidden[0]), 1)

        output = F.log_softmax(self.out(concat_output), dim=1)

        return output, hidden, attn_weights

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)


In [ ]:
teacher_forcing_ratio = 0.1


def train_attn(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=MAX_LENGTH):
    encoder_hidden = encoder.initHidden()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)

    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

    loss = 0

    for ei in range(min(input_length,max_length)):
        encoder_output, encoder_hidden = encoder(
            input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    decoder_input = torch.tensor([[SOS_token]], device=device)

    decoder_hidden = encoder_hidden

    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False

    if use_teacher_forcing:
        # Teacher forcing: Feed the target as the next input
        for di in range(min(target_length,max_length)):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]  # Teacher forcing

    else:
        # Without teacher forcing: use its own predictions as the next input
        for di in range(min(target_length,max_length)):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()  # detach from history as input

            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length


In [ ]:
def evaluate_attn(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()

        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(min(input_length,max_length)):
            encoder_output, encoder_hidden = encoder(input_tensor[ei],
                                                     encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

        decoder_hidden = encoder_hidden

        decoded_words = []
        decoder_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            decoder_attentions[di] = decoder_attention.data
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words, decoder_attentions[:di + 1]

In [ ]:
def evaluateRandomly_attn(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, attention= evaluate_attn(encoder, decoder, pair[0])
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [ ]:
def trainIters_attn(encoder, decoder, n_iters, print_every=1000, plot_every=100, learning_rate=0.01):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    training_pairs = [tensorsFromPair(random.choice(pairs))
                      for i in range(n_iters)]
    criterion = nn.NLLLoss()

    for iter in range(1, n_iters + 1):
        training_pair = training_pairs[iter - 1]
        input_tensor = training_pair[0]
        target_tensor = training_pair[1]

        loss = train_attn(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if iter % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, iter / n_iters),
                                         iter, iter / n_iters * 100, print_loss_avg))

        if iter % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)


In [ ]:
hidden_size = 256
encoder25 = EncoderRNN(input_lang.n_words, hidden_size).to(device)
attn_decoder25 = AttnDecoderRNN(hidden_size, output_lang.n_words, dropout_p=0.1).to(device)

#
n_iters = 15000
print_every = 1000

#
trainIters_attn(encoder25, attn_decoder25, n_iters= n_iters, print_every=print_every)

In [ ]:
evaluateRandomly_attn(encoder25, attn_decoder25)

In [ ]:
checkpoint2 = {
    'encoder_state': encoder25.state_dict(),
    'decoder_state': attn_decoder25.state_dict(),

}

torch.save(checkpoint2, 'seq2seq_checkpoint2.pth')

# Model 3 Stack more layers & try rnn/lstm

In [ ]:
class EncoderRNN1(nn.Module):
    def __init__(self, input_size, hidden_size,number_layers=1):
        super(EncoderRNN1, self).__init__()
        self.hidden_size = hidden_size
        self.number_layers = number_layers

        self.embedding = nn.Embedding(input_size, hidden_size)
        '''
        GRU is a gated RNN variant that captures long-term dependencies more effectively with fewer parameters. GRU has the same output shape as a standard RNN when configured identically.
        '''
        self.gru = nn.GRU(hidden_size, hidden_size,num_layers=self.number_layers)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output = embedded
        output, hidden = self.gru(output, hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(self.number_layers, 1, self.hidden_size, device=device)

In [ ]:
class DecoderRNN1(nn.Module):
    def __init__(self, hidden_size, output_size,number_layers=1):
        super(DecoderRNN1, self).__init__()
        self.hidden_size = hidden_size
        self.number_layers = number_layers

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size,num_layers = self.number_layers)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)


In [ ]:
hidden_size = 256
encoder11 = EncoderRNN1(input_lang.n_words, hidden_size,number_layers=3).to(device)
decoder11 = DecoderRNN1(hidden_size, output_lang.n_words, number_layers=3).to(device)

n_iters = 15000
print_every = 1000
trainIters(encoder11, decoder11, n_iters = n_iters, print_every=print_every)


In [ ]:
evaluateRandomly(encoder11, decoder11)

In [ ]:
class EncoderRNN2(nn.Module):
    def __init__(self, input_size, hidden_size,number_layers=1):
        super(EncoderRNN2, self).__init__()
        self.hidden_size = hidden_size
        self.number_layers = number_layers

        self.embedding = nn.Embedding(input_size, hidden_size)
        '''
        GRU is a gated RNN variant that captures long-term dependencies more effectively with fewer parameters. GRU has the same output shape as a standard RNN when configured identically.
        '''
        self.lstm = nn.LSTM(hidden_size, hidden_size,num_layers=self.number_layers)

    def forward(self, input, hidden):
        embedded = self.embedding(input).view(1, 1, -1)
        output = embedded
        output, hidden = self.lstm(output, hidden)
        return output, hidden

    def initHidden(self):
         return (torch.zeros(self.number_layers, 1, self.hidden_size, device=device),
                torch.zeros(self.number_layers, 1, self.hidden_size, device=device))


In [ ]:
class DecoderRNN2(nn.Module):
    def __init__(self, hidden_size, output_size,number_layers=1):
        super(DecoderRNN2, self).__init__()
        self.hidden_size = hidden_size
        self.number_layers = number_layers

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.lstm = nn.LSTM(hidden_size, hidden_size,num_layers = self.number_layers)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = F.relu(output)
        output, hidden = self.lstm(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
         return (torch.zeros(self.number_layers, 1, self.hidden_size, device=device),
                torch.zeros(self.number_layers, 1, self.hidden_size, device=device))



In [ ]:
hidden_size = 256
encoder12 = EncoderRNN2(input_lang.n_words, hidden_size,number_layers=3).to(device)
decoder12 = DecoderRNN2(hidden_size, output_lang.n_words, number_layers=3).to(device)

n_iters = 15000
print_every = 1000
trainIters(encoder12, decoder12, n_iters = n_iters, print_every=print_every)


In [ ]:
evaluateRandomly(encoder12, decoder12)

In [ ]:
checkpoint3 = {
    'encoder_state': encoder12.state_dict(),
    'decoder_state': decoder12.state_dict(),

}

torch.save(checkpoint3, 'seq2seq_checkpoint3.pth')



# Model 4 Using Pre-trained embedding

In [ ]:
import gensim.downloader as api
import numpy as np
import torch
import torch.nn as nn

class EncoderRNNWord2Vec(nn.Module):

    def __init__(
        self,
        lang_obj,
        hidden_size: int,
        embed_model_name: str = "glove-wiki-gigaword-100",
        freeze: bool = False,
        device: torch.device | str = "cpu",
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.device = torch.device(device)

        # Load pre-trained vectors
        print(f"Loading pre-trained vectors: {embed_model_name} …")
        word_vectors = api.load(embed_model_name)
        embed_dim = word_vectors.vector_size
        print(f"Loaded. Vector size = {embed_dim}")

        # Build embedding matrix aligned with lang_obj’s indices
        vocab_size = lang_obj.n_words
        matrix = np.random.normal(scale=0.01, size=(vocab_size, embed_dim))
        found, missing = 0, 0
        for token, idx in lang_obj.word2index.items():
            if token in word_vectors:
                matrix[idx] = word_vectors[token]
                found += 1
            else:
                missing += 1
        print(f"Matched {found}/{vocab_size} tokens; {missing} kept random.")

        # Create Embedding layer from that matrix
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(matrix, dtype=torch.float32),
            freeze=freeze,
        )


        self.gru = nn.GRU(embed_dim, hidden_size)


    def forward(self, input_tok: torch.Tensor, hidden: torch.Tensor):

        embedded = self.embedding(input_tok).view(1, 1, -1)  # (1, 1, embed_dim)
        output, hidden = self.gru(embedded, hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=self.device)


In [ ]:
class DecoderRNNWord2Vec(nn.Module):


    def __init__(
        self,
        lang_obj,
        hidden_size: int,
        embed_model_name: str = "glove-wiki-gigaword-100",
        freeze: bool = False,
        device: torch.device | str = "cpu",
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.device = torch.device(device)

        #   Load vectors
        print(f"Loading pre-trained vectors: {embed_model_name} …")
        word_vectors = api.load(embed_model_name)
        embed_dim = word_vectors.vector_size
        print(f"Loaded. Vector size = {embed_dim}")

        #   Build embedding matrix for the decoder vocab
        vocab_size = lang_obj.n_words
        matrix = np.random.normal(scale=0.01, size=(vocab_size, embed_dim))
        found = 0
        for token, idx in lang_obj.word2index.items():
            if token in word_vectors:
                matrix[idx] = word_vectors[token]
                found += 1
        print(f"Matched {found}/{vocab_size} decoder tokens.")

        # Replace the origin embedding to the new one
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(matrix, dtype=torch.float32),
            freeze=freeze,
        )


        self.gru = nn.GRU(embed_dim, hidden_size)


        self.out = nn.Linear(hidden_size, vocab_size)
        self.softmax = nn.LogSoftmax(dim=1)


    def forward(self, input_tok: torch.Tensor, hidden: torch.Tensor):

        emb = self.embedding(input_tok).view(1, 1, -1)  # (1,1,embed_dim)
        emb = F.relu(emb)
        output, hidden = self.gru(emb, hidden)          # output (1,1,hidden)
        output = self.softmax(self.out(output[0]))      # (1,vocab_size)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=self.device)


In [ ]:
hidden_size = 256

encoder14 = EncoderRNNWord2Vec(lang_obj=input_lang,
                             hidden_size=256,
                             embed_model_name="glove-wiki-gigaword-100",
                             freeze=False,
                             device="cuda").to(device)

decoder14 = DecoderRNNWord2Vec(
    lang_obj=output_lang,        # your target-side vocabulary helper
    hidden_size=256,
    embed_model_name="glove-wiki-gigaword-100",
    freeze=False,
    device="cuda",
).to(device)

n_iters = 15000
print_every = 1000
trainIters(encoder14, decoder14, n_iters = n_iters, print_every=print_every)


In [ ]:
evaluateRandomly(encoder14, decoder14)

In [ ]:
checkpoint4 = {
    'encoder_state': encoder14.state_dict(),
    'decoder_state': decoder14.state_dict(),

}

torch.save(checkpoint4, 'seq2seq_checkpoint4.pth')

# Model 4 plus : attention encoder with high teacher **rate**

In [ ]:
class AttnDecoderRNNWord2Vec(nn.Module):  # new name keeps the original untouched
    def __init__(
        self,
        lang_obj,
        hidden_size,
        dropout_p=0.1,
        max_length=MAX_LENGTH,
        embed_model_name="glove-wiki-gigaword-100",
        freeze=False,
        device: torch.device | str = "cpu",
    ):
        super().__init__()
        self.hidden_size = hidden_size
        self.output_size = lang_obj.n_words
        self.dropout_p  = dropout_p
        self.max_length = max_length
        self.device     = torch.device(device)


        print(f"Loading vectors: {embed_model_name} …")
        wv = api.load(embed_model_name)
        embed_dim = wv.vector_size
        matrix = np.random.normal(scale=0.01,
                                   size=(self.output_size, embed_dim))
        for token, idx in lang_obj.word2index.items():
            if token in wv:
                matrix[idx] = wv[token]
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(matrix, dtype=torch.float32),
            freeze=freeze,
        )

        self.dropout = nn.Dropout(self.dropout_p)
        self.gru     = nn.GRU(embed_dim, self.hidden_size)   # 🔄 embed_dim
        self.out     = nn.Linear(self.hidden_size * 2, self.output_size)


    def forward(self, input, hidden, encoder_outputs):
        embedded = self.embedding(input).view(1, 1, -1)
        embedded = self.dropout(embedded)

        _, hidden = self.gru(embedded, hidden)

        attn_weights = F.softmax(
            torch.bmm(hidden, encoder_outputs.T.unsqueeze(0)), dim=-1
        )                                              # (1,1,max_len)
        attn_output = torch.bmm(attn_weights,
                                encoder_outputs.unsqueeze(0))  # (1,1,hidden)

        concat_output = torch.cat((attn_output[0], hidden[0]), 1)
        output = F.log_softmax(self.out(concat_output), dim=1)
        return output, hidden, attn_weights

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=self.device)


In [ ]:
teacher_forcing_ratio1 = 0.9


def train_attn1(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=MAX_LENGTH):
    encoder_hidden = encoder.initHidden()

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)

    encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

    loss = 0

    for ei in range(min(input_length,max_length)):
        encoder_output, encoder_hidden = encoder(
            input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    decoder_input = torch.tensor([[SOS_token]], device=device)

    decoder_hidden = encoder_hidden

    use_teacher_forcing = True if random.random() < teacher_forcing_ratio1 else False

    if use_teacher_forcing:
        # Teacher forcing: Feed the target as the next input
        for di in range(min(target_length,max_length)):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]  # Teacher forcing

    else:
        # Without teacher forcing: use its own predictions as the next input
        for di in range(min(target_length,max_length)):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()  # detach from history as input

            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length


In [ ]:
def trainIters_attn1(encoder, decoder, n_iters, print_every=1000, plot_every=100, learning_rate=0.01):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
    training_pairs = [tensorsFromPair(random.choice(pairs))
                      for i in range(n_iters)]
    criterion = nn.NLLLoss()

    for iter in range(1, n_iters + 1):
        training_pair = training_pairs[iter - 1]
        input_tensor = training_pair[0]
        target_tensor = training_pair[1]

        loss = train_attn1(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if iter % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, iter / n_iters),
                                         iter, iter / n_iters * 100, print_loss_avg))

        if iter % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)


In [ ]:
hidden_size = 256

encoder15 = EncoderRNNWord2Vec(lang_obj=input_lang,
                             hidden_size=256,
                             embed_model_name="glove-wiki-gigaword-100",
                             freeze=False,
                             device="cuda").to(device)

decoder15 = AttnDecoderRNNWord2Vec(
    lang_obj     = output_lang,          # target vocabulary helper
    hidden_size  = 256,
    embed_model_name = "glove-wiki-gigaword-100",
    freeze       = False,
    device       = "cuda",
).to(device)

#
n_iters = 15000
print_every = 1000

#
trainIters_attn1(encoder15, decoder15, n_iters= n_iters, print_every=print_every)

In [ ]:
evaluateRandomly_attn(encoder15, decoder15)

In [ ]:
checkpoint5 = {
    'encoder_state': encoder15.state_dict(),
    'decoder_state': decoder15.state_dict(),

}

torch.save(checkpoint5, 'seq2seq_checkpoint5.pth')

# Part 5 model coverage mechanism

In [ ]:
class EncoderRNNS(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNNS, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)

    def forward(self, input, hidden):
        # embed and GRU step
        embedded = self.embedding(input).view(1, 1, -1)
        output, hidden = self.gru(embedded, hidden)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

In [ ]:


class DecoderRNNS(nn.Module):
    """
    Decoder with global attention and coverage mechanism
    """
    def __init__(self, hidden_size, output_size, max_length=MAX_LENGTH):
        super(DecoderRNNS, self).__init__()
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.max_length = max_length

        self.embedding = nn.Embedding(output_size, hidden_size)

        self.attn = nn.Linear(hidden_size * 2 + 1, 1)

        self.gru = nn.GRU(hidden_size * 2, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def initCoverage(self):

        return torch.zeros(self.max_length, device=device)

    def forward(self, input, hidden, encoder_outputs, coverage):

        embedded = self.embedding(input).view(1, 1, -1)

        # prepare encoder states
        enc = encoder_outputs[:self.max_length]

        # build decoder‐state matrix

        dec = hidden[0, 0].unsqueeze(0).repeat(self.max_length, 1)

        #  coverage vector (max_length, 1)
        cov = coverage.unsqueeze(1)


        #    attn_input (max_length, 2*H + 1)
        attn_input = torch.cat((enc, dec, cov), dim=1)
        energies     = self.attn(attn_input).squeeze(1)
        attn_weights = F.softmax(energies, dim=0)

        # coverage loss
        cov_loss = torch.sum(torch.min(attn_weights, coverage))
        coverage = coverage + attn_weights

        #  context vector

        context = torch.matmul(attn_weights.unsqueeze(0), enc.unsqueeze(0))

        # GRU step with [embedded; context]
        rnn_input = torch.cat((embedded, context), dim=2)
        output, hidden = self.gru(rnn_input, hidden)

        #  project to vocab
        output = self.log_softmax(self.out(output[0]))

        return output, hidden, attn_weights, coverage, cov_loss




In [ ]:
def trainS(input_tensor, target_tensor,
           encoder, decoder,
           encoder_optimizer, decoder_optimizer,
           criterion, cov_loss_weight=1.0):
    encoder_hidden = encoder.initHidden()
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    input_length = input_tensor.size(0)
    target_length = target_tensor.size(0)
    encoder_outputs = torch.zeros(MAX_LENGTH, encoder.hidden_size, device=device)
    loss = 0

    # encode
    for ei in range(min(input_length, MAX_LENGTH)):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
        encoder_outputs[ei] = encoder_output[0, 0]

    # decode
    decoder_input = torch.tensor([[SOS_token]], device=device)
    decoder_hidden = encoder_hidden
    coverage = decoder.initCoverage()
    use_tf = True if random.random() < 0.5 else False

    if use_tf:
        for di in range(min(target_length, MAX_LENGTH)):
            output, decoder_hidden, attn_w, coverage, cov_loss = decoder(
                 decoder_input, decoder_hidden, encoder_outputs, coverage)
            loss += criterion(output, target_tensor[di])
            loss += cov_loss_weight * cov_loss
            decoder_input = target_tensor[di].unsqueeze(0)
    else:
        for di in range(min(target_length, MAX_LENGTH)):
            output, decoder_hidden, attn_w, coverage, cov_loss = decoder(
                 decoder_input, decoder_hidden, encoder_outputs, coverage)
            topv, topi = output.topk(1)
            decoder_input = topi.detach()
            loss += criterion(output, target_tensor[di])
            loss += cov_loss_weight * cov_loss
            if decoder_input.item() == EOS_token:
                break

    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length

In [ ]:
def trainItersS(encoder, decoder,
                n_iters, print_every=1000,plot_every=100,
                learning_rate=0.01,
                cov_loss_weight=1.0):
    start = time.time()
    plot_losses = []
    print_loss_total = 0
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = torch.optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = torch.optim.SGD(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    training_pairs = [tensorsFromPair(random.choice(pairs)) for _ in range(n_iters)]

    for iteration in range(1, n_iters + 1):
        input_tensor, target_tensor = training_pairs[iteration - 1]
        loss = trainS(input_tensor, target_tensor,
                      encoder, decoder,
                      encoder_optimizer, decoder_optimizer,
                      criterion, cov_loss_weight)

        print_loss_total += loss
        plot_loss_total += loss

        if iteration % print_every == 0:
            avg_loss = print_loss_total / print_every
            print(f"{timeSince(start, iteration/n_iters)} ({iteration}/{n_iters}) {avg_loss:.4f}")
            print_loss_total = 0

        if iteration % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [ ]:
# Example setup
hidden_size = 256
encoder21 = EncoderRNNS(input_lang.n_words, hidden_size).to(device)
decoder21 = DecoderRNNS(hidden_size, output_lang.n_words).to(device)

n_iters = 15000
print_every = 1000
trainItersS(encoder21, decoder21, n_iters=n_iters, print_every=print_every)

In [ ]:
checkpoint6 = {
    'encoder_state': encoder21.state_dict(),
    'decoder_state': decoder21.state_dict(),

}

torch.save(checkpoint6, 'seq2seq_checkpoint6.pth')

# Part 6 process ingredient as a set

In [ ]:
class SetEncoder(nn.Module):


    def __init__(self, input_size, hidden_size, dropout_p: float = 0.1):
        super().__init__()
        self.hidden_size = hidden_size

        # 1 READ
        self.embedding = nn.Embedding(input_size, hidden_size)

        # 2) PROCESS
        self.phi = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_p)
        )

        # 3 WRITE
        self.rho = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh()
        )

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

    def forward(self, input_seq):

        emb   = self.embedding(input_seq).squeeze(1)   # (S, H)
        phi_x = self.phi(emb)                          # (S, H)
        pooled = phi_x.mean(dim=0)
        h = self.rho(pooled).unsqueeze(0).unsqueeze(1) # (1, 1, H)
        return h




In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(output_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        output = self.embedding(input).view(1, 1, -1)
        output = F.relu(output)
        output, hidden = self.gru(output, hidden)
        output = self.softmax(self.out(output[0]))
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, 1, self.hidden_size, device=device)

teacher_forcing_ratio = 0.5

In [ ]:
def train(input_tensor, target_tensor, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion, max_length=MAX_LENGTH):
    encoder_hidden = encoder(input_tensor)

    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()

    target_length = target_tensor.size(0)
    loss = 0

    decoder_input = torch.tensor([[SOS_token]], device=device)

    decoder_hidden = encoder_hidden

    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False



    if use_teacher_forcing:
        # Teacher forcing: Feed the target as the next input

        for di in range(min(target_length,max_length)):

            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden)

            loss += criterion(decoder_output, target_tensor[di])


            decoder_input = target_tensor[di]  # Teacher forcing

    else:
        # Without teacher forcing: use its own predictions as the next input
        for di in range(min(target_length,max_length)):
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()  # detach from history as input

            loss += criterion(decoder_output, target_tensor[di])
            if decoder_input.item() == EOS_token:
                break

    loss.backward()

    encoder_optimizer.step()
    decoder_optimizer.step()

    return loss.item() / target_length

In [ ]:
def trainIters(encoder, decoder, n_iters, print_every=1000, plot_every=100, learning_rate=0.01):

    # this part is for set-up

    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)

    training_pairs = [tensorsFromPair(random.choice(pairs))
                      for i in range(n_iters)]
    criterion = nn.NLLLoss()


    # this part is for the main training loop

    for iter in range(1, n_iters + 1):                                              # n_iters means the total number of training iterations (or steps) you want to run.

        training_pair = training_pairs[iter - 1]

        input_tensor = training_pair[0]
        target_tensor = training_pair[1]                                            # target_tensor is the tensor representation of the correct output sentence.

        loss = train(input_tensor, target_tensor, encoder,
                     decoder, encoder_optimizer, decoder_optimizer, criterion)      # here is one step to using train.

        print_loss_total += loss
        plot_loss_total += loss

        if iter % print_every == 0:

            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, iter / n_iters),
                                         iter, iter / n_iters * 100, print_loss_avg))

        if iter % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [ ]:
hidden_size = 256
encoder39   = SetEncoder(input_lang.n_words, hidden_size).to(device)
decoder39   = DecoderRNN(hidden_size, output_lang.n_words).to(device)


n_iters = 15000
print_every = 1000
trainIters(encoder39, decoder39, n_iters = n_iters, print_every=print_every)

# Evaluation

In [ ]:
!gdown --fuzzy https://drive.google.com/file/d/1b1ukR6pxWGVaicGVu719_u-sJrkSr-3L/view?usp=sharing

In [ ]:
def prepareData_EV(file_path, src_col, tgt_col, reverse=False):
    """
    1. Load & normalize with readCSV
    2. Print original size
    3. Filter by MAX_LENGTH
    4. Build vocab with Lang.addSentence
    5. Print trimmed size & vocab counts
    """
    # 1. load
    input_lang, output_lang, pairs = readCSV(
    file_path  = "/content/test.csv",
    src_col    = "Ingredients",
    tgt_col    = "Recipe",
    reverse    = False
)
    print(f"Read {len(pairs)} sentence pairs")



    # 3. build vocab
    print("Counting words...")
    for src, tgt in pairs:
        input_lang.addSentence(src)
        output_lang.addSentence(tgt)

    # 4. report
    print("Counted words:")
    print(f"{input_lang.name}: {input_lang.n_words}")
    print(f"{output_lang.name}: {output_lang.n_words}")
    return input_lang, output_lang, pairs

input_lang_EV, output_lang_EV, pairs_EV = prepareData_EV(file_path, src_col, tgt_col)
print(random.choice(pairs))

In [ ]:
!pip install nltk bert-score -q

In [ ]:
import nltk
nltk.download('punkt', quiet=True)   # tokeniser
nltk.download('wordnet', quiet=True) # METEOR
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt_tab')


from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from bert_score import score as bert_score

smoothie = SmoothingFunction().method4   # BLEU smoothing

## This one is model 3 & basic line model 1

In [ ]:
def evaluate_EV(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang_EV, sentence)
        input_length = input_tensor.size()[0]

        encoder_hidden = encoder.initHidden()

        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(min(input_length,max_length)):
            encoder_output, encoder_hidden = encoder(input_tensor[ei],
                                                     encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

        decoder_hidden = encoder_hidden

        decoded_words = []
        decoder_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden)
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words

In [ ]:
def evaluate_corpus(
    encoder,
    decoder,
    pairs,                   # list of (src, tgt) tuples   ← pairs_EV
    input_lang,
    output_lang,
    max_length=MAX_LENGTH,
    device=device,
    verbose=False            # True = print each translation
):
    gold_texts, pred_texts   = [], []

    for idx, (src, tgt) in enumerate(pairs):



        pred_words = evaluate_EV(encoder, decoder, src, max_length)
        pred_sentence = " ".join(w for w in pred_words if w != "<EOS>")

        gold_texts.append(tgt)
        pred_texts.append(pred_sentence)

        if verbose:
            print(f"[{idx:>4}] SRC: {src}")
            print(f"      REF: {tgt}")
            print(f"      HYP: {pred_sentence}\n")


    bleu_scores, meteor_scores = [], []

    for ref, hyp in zip(gold_texts, pred_texts):

        ref_tok  = nltk.word_tokenize(ref)
        hyp_tok  = nltk.word_tokenize(hyp)

        bleu_scores.append(
            sentence_bleu([ref_tok], hyp_tok,
                          weights=(0.25, 0.25, 0.25, 0.25),
                          smoothing_function=smoothie)
        )

        meteor_scores.append(meteor_score([ref_tok], hyp_tok))

    bleu_avg   = sum(bleu_scores)   / len(bleu_scores)
    meteor_avg = sum(meteor_scores) / len(meteor_scores)

    #  BERTScore
    _, _, f1 = bert_score(pred_texts, gold_texts, lang="en", verbose=False)
    bert_f1 = f1.mean().item()

    print("───────────────────────────────────────────")
    print(f"Evaluated {len(pairs)} test sentences")
    print(f"BLEU-4   : {bleu_avg:.4f}")
    print(f"METEOR   : {meteor_avg:.4f}")
    print(f"BERTScore: {bert_f1:.4f}")

    return bleu_avg, meteor_avg, bert_f1,pred_texts

In [ ]:
!gdown --fuzzy https://drive.google.com/file/d/1D0EJF0-RFbI92rpe_1a3iCKijYYkwCXV/view?usp=sharing

In [ ]:
ckpt = torch.load('seq2seq_checkpoint1.pth', map_location=device)
encoder11.load_state_dict(ckpt['encoder_state'])
decoder11.load_state_dict(ckpt['decoder_state'])

In [ ]:
bleu, meteor, bert_f1,pred_texts_1 = evaluate_corpus(
    encoder         = encoder11,
    decoder         = decoder11,
    pairs           = pairs_EV,         # full test set
    input_lang      = input_lang_EV,
    output_lang     = output_lang_EV,
    device          = "cuda"
)

In [ ]:
import pandas as pd


df = pd.DataFrame({"Recipe - Baseline 1": pred_texts_1})


df.to_excel("predictions_model_1_plus.xlsx",
            index=False,
            sheet_name="Baseline_1")



In [ ]:

raw_sentences = [
    "sugar, lemon juice,  water,  orange juice, strawberries, icecream",
    "8 oz philadelphia cream cheese, 14 oz can sweetened condensed milk, "
    "1 ts vanilla, 1/3 c  lemon juice, 48 oz canned cherries, "
    "8 inch graham cracker,  pie crusts",
]


for i, raw in enumerate(raw_sentences, 1):
    test_sentence = normalizeString(raw)

    decoded_tokens  = evaluate_EV(
        encoder     = encoder11,
        decoder     = decoder11,
        sentence    = test_sentence,
        max_length  = MAX_LENGTH,
    )

    recipe = " ".join(tok for tok in decoded_tokens if tok != "<EOS>")
    print(f"Generated recipe {i}:\n{recipe}\n" + "-"*60)


##Model 3

In [ ]:
bleu, meteor, bert_f1,pred_texts_3 = evaluate_corpus(
    encoder         = encoder11,
    decoder         = decoder11,
    pairs           = pairs_EV,
    input_lang      = input_lang_EV,
    output_lang     = output_lang_EV,
    device          = "cuda"
)

In [ ]:
import pandas as pd


df = pd.DataFrame({"Recipe - 3": pred_texts_3})


df.to_excel("predictions_model_3_plus.xlsx",
            index=False,
            sheet_name="Baseline_1")



## This one is model2 & model 4 plus

In [ ]:
def evaluate_EV_attn(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang_EV, sentence)
        input_length = input_tensor.size()[0]
        encoder_hidden = encoder.initHidden()

        encoder_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)

        for ei in range(min(input_length,max_length)):
            encoder_output, encoder_hidden = encoder(input_tensor[ei],
                                                     encoder_hidden)
            encoder_outputs[ei] += encoder_output[0, 0]

        decoder_input = torch.tensor([[SOS_token]], device=device)  # SOS

        decoder_hidden = encoder_hidden

        decoded_words = []
        decoder_attentions = torch.zeros(max_length, max_length)

        for di in range(max_length):
            decoder_output, decoder_hidden, decoder_attention = decoder(
                decoder_input, decoder_hidden, encoder_outputs)
            decoder_attentions[di] = decoder_attention.data
            topv, topi = decoder_output.data.topk(1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words, decoder_attentions[:di + 1]

In [ ]:
def evaluate_corpus_attn(
    encoder,
    decoder,
    pairs,                   # list of (src, tgt) tuples
    input_lang,
    output_lang,
    max_length=MAX_LENGTH,
    device=device,
    verbose=False
):
    gold_texts, pred_texts   = [], []

    for idx, (src, tgt) in enumerate(pairs):
        pred_words, _ = evaluate_EV_attn(encoder, decoder, src, max_length)
        pred_sentence = " ".join(w for w in pred_words if w != "<EOS>")

        gold_texts.append(tgt)
        pred_texts.append(pred_sentence)

        if verbose:
            print(f"[{idx:>4}] SRC: {src}")
            print(f"      REF: {tgt}")
            print(f"      HYP: {pred_sentence}\n")

    #  sentence-level BLEU / METEOR
    bleu_scores, meteor_scores = [], []

    for ref, hyp in zip(gold_texts, pred_texts):

        ref_tok  = nltk.word_tokenize(ref)
        hyp_tok  = nltk.word_tokenize(hyp)

        bleu_scores.append(
            sentence_bleu([ref_tok], hyp_tok,
                          weights=(0.25, 0.25, 0.25, 0.25),
                          smoothing_function=smoothie)
        )

        meteor_scores.append(meteor_score([ref_tok], hyp_tok))

    bleu_avg   = sum(bleu_scores)   / len(bleu_scores)
    meteor_avg = sum(meteor_scores) / len(meteor_scores)

    #  corpus-level BERTScore
    _, _, f1 = bert_score(pred_texts, gold_texts, lang="en", verbose=False)
    bert_f1 = f1.mean().item()

    print("───────────────────────────────────────────")
    print(f"Evaluated {len(pairs)} test sentences")
    print(f"BLEU-4   : {bleu_avg:.4f}")
    print(f"METEOR   : {meteor_avg:.4f}")
    print(f"BERTScore: {bert_f1:.4f}")

    return bleu_avg, meteor_avg, bert_f1,pred_texts


In [ ]:
bleu, meteor, bert_f1, pred_texts_4 = evaluate_corpus_attn(
    encoder         = encoder15,
    decoder         = decoder15,
    pairs           = pairs_EV,         # full test set
    input_lang      = input_lang_EV,
    output_lang     = output_lang_EV,
    device          = "cuda"
)


In [ ]:
import pandas as pd


df = pd.DataFrame({"Recipe - Baseline 1": pred_texts_4})


df.to_excel("predictions_model_4_plus.xlsx",
            index=False,
            sheet_name="Baseline_1")



### model 2

In [ ]:
bleu, meteor, bert_f1, pred_texts_model_2 = evaluate_corpus_attn(
    encoder         = encoder25,
    decoder         = attn_decoder25,
    pairs           = pairs_EV,
    input_lang      = input_lang_EV,
    output_lang     = output_lang_EV,
    device          = "cuda"       )


In [ ]:
import pandas as pd


df = pd.DataFrame({"Recipe - Baseline 1": pred_texts_model_2})


df.to_excel("predictions_model_2.xlsx",
            index=False,
            sheet_name="Baseline_1")



## This one is Model 5

In [ ]:
def evaluate_EV_attn_coverage(encoder, decoder, sentence,
                              input_lang, output_lang,
                              max_length=MAX_LENGTH, device=device):
    """
    Runs the encoder + coverage-enabled decoder on one sentence.
    Returns (decoded_words_list, attention_matrix).
    """
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        # Encode
        input_tensor = tensorFromSentence(input_lang, sentence).to(device)
        input_len = input_tensor.size(0)
        enc_hidden = encoder.initHidden()
        enc_outputs = torch.zeros(max_length, encoder.hidden_size, device=device)
        for i in range(min(input_len, max_length)):
            out, enc_hidden = encoder(input_tensor[i], enc_hidden)
            enc_outputs[i] = out[0, 0]

        # Prepare decoder inputs
        dec_input = torch.tensor([[SOS_token]], device=device)
        dec_hidden = enc_hidden
        coverage_vec = decoder.initCoverage()
        decoded_words = []
        attn_matrix = torch.zeros(max_length, max_length, device=device)

        # Decode loop
        for t in range(max_length):
            out, dec_hidden, attn_w, coverage_vec, _ = decoder(
                dec_input, dec_hidden, enc_outputs, coverage_vec
            )
            attn_matrix[t, :attn_w.size(0)] = attn_w
            topv, topi = out.topk(1)
            token_id = topi.item()
            if token_id == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[token_id])
            dec_input = topi.detach().unsqueeze(0)

        return decoded_words, attn_matrix[:t+1]

In [ ]:
def evaluate_corpus_attn_coverage(encoder, decoder, pairs,
                                  input_lang, output_lang,
                                  max_length=MAX_LENGTH, device=device,
                                  verbose=False):
    """
    Runs evaluate_EV_attn_coverage on each (src, tgt) in pairs,
    computes BLEU-4, METEOR, and BERTScore F1, and returns metrics + hypotheses.
    """
    refs, hyps = [], []
    for idx, (src, tgt) in enumerate(pairs):
        pred_tokens, _ = evaluate_EV_attn_coverage(
            encoder, decoder, src,
            input_lang, output_lang,
            max_length, device
        )
        hypo = " ".join(w for w in pred_tokens if w != '<EOS>')
        refs.append(tgt)
        hyps.append(hypo)
        if verbose:
            print(f"[{idx:>4}] SRC: {src}")
            print(f"      REF: {tgt}")
            print(f"      HYP: {hypo}\n")

    # Sentence-level metrics
    bleu_scores = []
    meteor_scores = []
    for r, h in zip(refs, hyps):
        r_tok = nltk.word_tokenize(r)
        h_tok = nltk.word_tokenize(h)
        bleu_scores.append(
            sentence_bleu([r_tok], h_tok,
                          weights=(0.25,)*4,
                          smoothing_function=smoothie)
        )
        meteor_scores.append(meteor_score([r_tok], h_tok))

    bleu_avg = sum(bleu_scores) / len(bleu_scores)
    meteor_avg = sum(meteor_scores) / len(meteor_scores)

    # BERTScore
    _, _, f1_scores = bert_score(hyps, refs, lang='en', verbose=False)
    bert_f1 = f1_scores.mean().item()

    print("─"*40)
    print(f"Evaluated {len(pairs)} sentences")
    print(f"BLEU-4   : {bleu_avg:.4f}")
    print(f"METEOR   : {meteor_avg:.4f}")
    print(f"BERTScore: {bert_f1:.4f}")

    return bleu_avg, meteor_avg, bert_f1, hyps

In [ ]:
bleu, meteor, bert_f1, pred_texts_5 = evaluate_corpus_attn_coverage(
    encoder      = encoder21,
    decoder      = decoder21,
    pairs        = pairs_EV,
    input_lang   = input_lang_EV,
    output_lang  = output_lang_EV,
    device       = device,
    verbose      = False
)



In [ ]:
import pandas as pd


df = pd.DataFrame({"Recipe - Baseline 1": pred_texts_5})


df.to_excel("predictions_model_5.xlsx",
            index=False,
            sheet_name="Baseline_1")



# this one is model 6

In [ ]:
def evaluate_EV(encoder, decoder, sentence, max_length=MAX_LENGTH):
    with torch.no_grad():
        input_tensor   = tensorFromSentence(input_lang_EV, sentence)
        encoder_hidden = encoder(input_tensor)

        decoder_input  = torch.tensor([[SOS_token]], device=device)
        decoder_hidden = encoder_hidden

        decoded_words = []

        for _ in range(max_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.topk(1)         # (1,1)
            if topi.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[topi.item()])

            decoder_input = topi.squeeze().detach()

        return decoded_words


def evaluate_corpus(
    encoder,
    decoder,
    pairs,                   # list of (src, tgt) tuples   ← pairs_EV
    input_lang,
    output_lang,
    max_length=MAX_LENGTH,
    device=device,
    verbose=False
):
    gold_texts, pred_texts   = [], []

    for idx, (src, tgt) in enumerate(pairs):



        pred_words = evaluate_EV(encoder, decoder, src, max_length)
        pred_sentence = " ".join(w for w in pred_words if w != "<EOS>")

        gold_texts.append(tgt)
        pred_texts.append(pred_sentence)

        if verbose:
            print(f"[{idx:>4}] SRC: {src}")
            print(f"      REF: {tgt}")
            print(f"      HYP: {pred_sentence}\n")

    # ---- sentence-level BLEU / METEOR, then average ----
    bleu_scores, meteor_scores = [], []

    for ref, hyp in zip(gold_texts, pred_texts):

        ref_tok  = nltk.word_tokenize(ref)
        hyp_tok  = nltk.word_tokenize(hyp)

        bleu_scores.append(
            sentence_bleu([ref_tok], hyp_tok,
                          weights=(0.25, 0.25, 0.25, 0.25),
                          smoothing_function=smoothie)
        )

        meteor_scores.append(meteor_score([ref_tok], hyp_tok))

    bleu_avg   = sum(bleu_scores)   / len(bleu_scores)
    meteor_avg = sum(meteor_scores) / len(meteor_scores)

    #  corpus-level BERTScore
    _, _, f1 = bert_score(pred_texts, gold_texts, lang="en", verbose=False)
    bert_f1 = f1.mean().item()

    print("───────────────────────────────────────────")
    print(f"Evaluated {len(pairs)} test sentences")
    print(f"BLEU-4   : {bleu_avg:.4f}")
    print(f"METEOR   : {meteor_avg:.4f}")
    print(f"BERTScore: {bert_f1:.4f}")

    return bleu_avg, meteor_avg, bert_f1,pred_texts

bleu, meteor, bert_f1, pred_texts_6 = evaluate_corpus(
    encoder      = encoder39,
    decoder      = decoder39,
    pairs        = pairs_EV,
    input_lang   = input_lang_EV,
    output_lang  = output_lang_EV,
    device       = device
)

import pandas as pd


df = pd.DataFrame({"Recipe - Baseline 1": pred_texts_6})


df.to_excel("predictions_model_6.xlsx",
            index=False,
            sheet_name="Baseline_1")

